In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn_extra.cluster import KMedoids
from sklearn.datasets import make_blobs
import pandas as pd
from math import radians, sin, cos, sqrt, atan2
from sklearn.metrics import pairwise_distances
import time
import os
from pathlib import Path

**Subtareas**
- Generar una función que permita calcular la matriz de distancias o el conjunto de coordenadas en un estado dado. Esto para cualquier método de clustering posterior.

In [3]:
original = pd.read_parquet("../2. cleaning_loads/parquetloadsfull.parquet")
original.shape

(57573489, 20)

In [4]:
carpeta = Path("coordenadas")  # Reemplaza con la ruta correcta
#archivos_parquet = list(carpeta.glob("*.parquet"))
#df = pd.concat([pd.read_parquet(archivo) for archivo in archivos_parquet], ignore_index=True)
#df = df.drop_duplicates(subset=['ID'])

In [5]:
columns = ['ID', 'Posted', 'PickUp', 'CityOrigin',
 'CityDestination', 'Distance', 'Source', 'Equip', 'StateOrigin',
 'HubOrigin', 'StateDestination', 'HubDestination', 'Broker_Shipper',
 'RatePerMile', 'Weight', 'Size', 'LatOrigin', 'LngOrigin', 'LatDestination', 'LngDestination']
#df = df[columns]

In [6]:
def haversine(lat1, lon1, lat2, lon2):
    """Calculate the Haversine distance between two points on the Earth."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = 3958.8 * c  # Radius of the Earth in miles
    return distance

### 1. Limpieza de coordenadas

In [7]:
us_state_codes = ['AL',"AZ", "AR", 
    "CA", "CO", "CT", "DE", "FL", "GA",
    "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"
]

In [8]:
state_code_dict = {    'Alabama': 'AL',    'Alaska': 'AK',    'Arizona': 'AZ',    'Arkansas': 'AR',    'California': 'CA',
    'Colorado': 'CO',    'Connecticut': 'CT',    'Delaware': 'DE',    'District of Columbia': 'DC',
    'Florida': 'FL',    'Georgia': 'GA',    'Hawaii': 'HI',    'Idaho': 'ID',    'Illinois': 'IL',    'Indiana': 'IN',    'Iowa': 'IA',    'Kansas': 'KS',
    'Kentucky': 'KY',    'Louisiana': 'LA',    'Maine': 'ME',    'Maryland': 'MD',
    'Massachusetts': 'MA',    'Michigan': 'MI',    'Minnesota': 'MN',    'Mississippi': 'MS',    'Missouri': 'MO',
    'Montana': 'MT',    'Nebraska': 'NE',    'Nevada': 'NV',    'New Hampshire': 'NH',    'New Jersey': 'NJ',    'New Mexico': 'NM',
    'New York': 'NY',    'North Carolina': 'NC',    'North Dakota': 'ND',
    'Ohio': 'OH',    'Oklahoma': 'OK',    'Oregon': 'OR',    'Pennsylvania': 'PA',    'Rhode Island': 'RI',    'South Carolina': 'SC',
    'South Dakota': 'SD',    'Tennessee': 'TN',    'Texas': 'TX',    'Utah': 'UT',    'Vermont': 'VT',    'Virginia': 'VA',    'Washington': 'WA',    'West Virginia': 'WV',
    'Wisconsin': 'WI',    'Wyoming': 'WY'
}

In [9]:
def mapa_estado(coords):
    usa_map = folium.Map(location=[39.5, -98.35], zoom_start=4)
    
    for _, row in coords.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['lng']],
            radius=4,  # Tamaño del punto
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=f"Lat: {row['lat']}, Lng: {row['lng']}, city: {row['city']}"
        ).add_to(usa_map)
    
    return usa_map

import folium
#limpiar coordenadas que están fuera del estado

In [10]:
def coordenadas(test,state):
    origins = test.loc[test['StateOrigin']==state][['LatOrigin','LngOrigin','StateOrigin','CityOrigin']].rename(columns = {'LatOrigin':'lat','LngOrigin':'lng','StateOrigin':'state','CityOrigin':'city'})
    destinations = test.loc[test['StateDestination']==state][['LatDestination','LngDestination','StateDestination','CityDestination']].rename(columns = {'LatDestination':'lat','LngDestination':'lng', 'StateDestination':'state','CityDestination':'city'})
    coords = pd.concat([origins,destinations])
    coords = coords.drop_duplicates(subset=['lat','lng'])
    coords = coords.dropna(subset=['lat','lng'])
    return coords
#solo deben salir las coordenadas que están dentro de, tanto destinos como origenes

In [11]:
#Encontrar el estado al que pertenece de cada carga
import geopandas as gpd
from shapely.geometry import Point

def estados_correctos(coords):
    # Supongamos que coords es tu DataFrame con columnas 'lat' y 'lng'
    geometry = [Point(xy) for xy in zip(coords['lng'], coords['lat'])]
    points = gpd.GeoDataFrame(coords, geometry=geometry, crs="EPSG:4326")  # Asegura el CRS correcto
    
    # Cargar geometría de estados
    states = gpd.read_file("https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json")
    
    # Spatial join para encontrar en qué estado cae cada punto
    joined = gpd.sjoin(points, states, how="left", predicate="within")
    
    # Crear nueva columna en el DataFrame original con el nombre del estado
    coords['newstate'] = joined['name']
    coords['newstate'] = coords['newstate'].replace(state_code_dict)
    coords['mix_state'] = coords.apply(lambda row: row['state'] if row['state'] == row['newstate'] or pd.isna(row['newstate']) else row['newstate'],axis=1)
    
    return coords, joined

In [12]:
def limpieza_dforigin(coords,df,state):
    changes = coords.loc[(coords['newstate']!=state)&(~coords['newstate'].isna())][['lat','lng','mix_state']]
    df = df.merge(changes, right_on =['lat','lng'], left_on = ['LatOrigin','LngOrigin'], how = 'left').drop(columns = ['lat','lng'])
    print(df.columns.tolist())
    df = df.merge(changes.rename(columns={'mix_state':'state'}), right_on =['lat','lng'], left_on = ['LatDestination','LngDestination'], how = 'left').drop(columns = ['lat','lng'])
    print(df.columns.tolist())
    df.loc[df['mix_state'].isna(),'mix_state'] = df['state']
    if state == 'CT':
        df.loc[df['mix_state']=='RI','mix_state'] = 'CT'    
    df.loc[(df['StateDestination']==state)&(df['mix_state'].notna())|(df['state'].notna()),'StateDestination'] = df['mix_state']
    df.loc[(df['StateOrigin']==state)&(df['mix_state'].notna())|(df['state'].notna()),'StateOrigin'] = df['mix_state']
    df = df[columns]
    return df

In [13]:
us_state_codes = ['AL',"AZ", "AR", 
    "CA", "CO", "CT", "DE", "FL", "GA",
    "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR"]
us_state_codes = ["PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY"
]

In [15]:
g1 = ['AL',"AZ", "AR"]
g2 = ["ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD", "MA"]
g3 = ["MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ"]
g4 = ["NM", "NY", "NC", "ND", "OH", "OK", "OR", 'PA']
g5 = ["RI", "SC","SD", "TN", "TX", "UT"] 
g6 = ["VT", "VA", "WA", "WV", "WI", "WY"]
g7 = ["CA", "CO", "CT", "DE"]
g8 = ["FL", "GA"]

In [19]:
#Para limpieza preliminar del estado, va en cada grupo iterando
for group in [g8]:
    for state in group:
        print(state)
        t = original.loc[(original['StateOrigin']==state)|(original['StateDestination']==state)]
        coords = coordenadas(t,state)
        #mapa = mapa_estado(coords)
        #display(mapa)
        coords,joined = estados_correctos(coords)
        t = limpieza_dforigin(coords,t,state)
    #   print('despues de tocar df original',prueba.shape)
        coords = coordenadas(t,state)
        #mapa = mapa_estado(coords)
        #display(mapa)
        t.to_parquet(f'coordenadas/{state}.parquet')

FL
['ID', 'Posted', 'PickUp', 'CityOrigin', 'LatOrigin', 'LngOrigin', 'CityDestination', 'LatDestination', 'LngDestination', 'Distance', 'Source', 'Equip', 'StateOrigin', 'HubOrigin', 'StateDestination', 'HubDestination', 'RatePerMile', 'Weight', 'Size', 'Broker_Shipper', 'mix_state']
['ID', 'Posted', 'PickUp', 'CityOrigin', 'LatOrigin', 'LngOrigin', 'CityDestination', 'LatDestination', 'LngDestination', 'Distance', 'Source', 'Equip', 'StateOrigin', 'HubOrigin', 'StateDestination', 'HubDestination', 'RatePerMile', 'Weight', 'Size', 'Broker_Shipper', 'mix_state', 'state']
GA
['ID', 'Posted', 'PickUp', 'CityOrigin', 'LatOrigin', 'LngOrigin', 'CityDestination', 'LatDestination', 'LngDestination', 'Distance', 'Source', 'Equip', 'StateOrigin', 'HubOrigin', 'StateDestination', 'HubDestination', 'RatePerMile', 'Weight', 'Size', 'Broker_Shipper', 'mix_state']
['ID', 'Posted', 'PickUp', 'CityOrigin', 'LatOrigin', 'LngOrigin', 'CityDestination', 'LatDestination', 'LngDestination', 'Distance', 'S

### 2. Limpiezas manuales

In [21]:
#Hay mapas de estados que tienen coordenadas pintadas fuera

In [22]:
#Uso de diccionario de coordenadas para aplicarlo manualmente por estados con problemas, esto se hizo buscando en maps los nombres de la ciudad y sus verdaderas coordenadas
#y la correspondencia con los estados.
ajustes = pd.read_excel('ajustes.xlsx')
ajustes.shape

(199, 5)

In [23]:
ajustes['latm']=ajustes['latm'].astype(float)
ajustes['lngm']=ajustes['lngm'].astype(float)
ajustes['latb']=ajustes['latb'].astype(float)
ajustes['lngb']=ajustes['lngb'].astype(float)
ajustes.info()

<class 'pandas.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   latm    199 non-null    float64
 1   lngm    199 non-null    float64
 2   latb    199 non-null    float64
 3   lngb    199 non-null    float64
 4   stateb  199 non-null    str    
dtypes: float64(4), str(1)
memory usage: 8.3 KB


In [24]:
ajustes = ajustes.drop_duplicates(subset =['latm','lngm'])

In [264]:
state= 'WY'
tt = pd.read_parquet(f'coordenadas/{state}.parquet')
coords = coordenadas(tt,state)
print(tt.shape)
print(coords.shape)
mapa = mapa_estado(coords)
display(mapa)

#documentar que la limpieza se hace viendo el mapa y viendo coordenadas por fuera del estado y agregándolas al excel de ajustes una a una

(286621, 20)
(2237, 4)


In [265]:
# Redondear coordenadas para evitar problemas de precisión float
DECIMALES = 5

tt_merged = tt.copy()
tt_merged['_lat_key'] = tt_merged['LatOrigin'].round(DECIMALES)
tt_merged['_lng_key'] = tt_merged['LngOrigin'].round(DECIMALES)

ajustes_merged = ajustes.copy()
ajustes_merged['_lat_key'] = ajustes_merged['latm'].round(DECIMALES)
ajustes_merged['_lng_key'] = ajustes_merged['lngm'].round(DECIMALES)

# Merge usando las claves redondeadas
df_loads = tt_merged.merge(
    ajustes_merged,
    how='left',
    on=['_lat_key', '_lng_key'],
    suffixes=('', '_fix')
)

# Si hay coordenadas buenas (latb/lngb), reemplazamos
df_loads['LatOrigin'] = df_loads['latb'].combine_first(df_loads['LatOrigin'])
df_loads['LngOrigin'] = df_loads['lngb'].combine_first(df_loads['LngOrigin'])
df_loads['StateOrigin'] = df_loads['stateb'].combine_first(df_loads['StateOrigin'])

# Limpiamos columnas auxiliares
df_loads.drop(columns=['latm', 'lngm', 'latb', 'lngb', 'stateb', '_lat_key', '_lng_key'], inplace=True)

In [266]:
# Claves redondeadas para destination
df_loads['_lat_key'] = df_loads['LatDestination'].round(DECIMALES)
df_loads['_lng_key'] = df_loads['LngDestination'].round(DECIMALES)

ajustes_dest = ajustes.copy()
ajustes_dest['_lat_key'] = ajustes_dest['latm'].round(DECIMALES)
ajustes_dest['_lng_key'] = ajustes_dest['lngm'].round(DECIMALES)

# Merge con coordenadas malas en destino
df_loads = df_loads.merge(
    ajustes_dest,
    how='left',
    on=['_lat_key', '_lng_key'],
    suffixes=('', '_fix')
)

# Si hay coordenadas buenas (latb/lngb), reemplazamos
df_loads['LatDestination'] = df_loads['latb'].combine_first(df_loads['LatDestination'])
df_loads['LngDestination'] = df_loads['lngb'].combine_first(df_loads['LngDestination'])
df_loads['StateDestination'] = df_loads['stateb'].combine_first(df_loads['StateDestination'])

# Limpiamos columnas auxiliares
df_loads.drop(columns=['latm', 'lngm', 'latb', 'lngb', 'stateb', '_lat_key', '_lng_key'], inplace=True)

In [267]:
coords = coordenadas(df_loads,state)
mapa = mapa_estado(coords)
display(mapa)

In [268]:
df_loads.to_parquet(f'coordenadas/{state}.parquet')